<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/18_classification/18_2_LogisticRegression/18_2_9_LogReg_Exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classification: Logistic Regression
## Exercise — Credit Risk with Interpretability Constraints

**Author:** Brad Sheese

---

## Context

You are a data scientist at a bank. The compliance team has told you that any model used to make credit decisions must be able to explain individual rejections to applicants — a legal requirement under the Equal Credit Opportunity Act.

This means XGBoost is off the table. You must use logistic regression.

Your task is to build, evaluate, and interpret a logistic regression model on the South German Credit dataset — the same dataset from Module 18_1. This time, the emphasis is on *what you can say about the model*, not just how accurate it is.

### What You Will Build

1. A logistic regression pipeline with correct preprocessing
2. A forest plot of odds ratios with confidence intervals
3. A plain-language explanation of one individual prediction
4. A model comparison table: LR vs. XGBoost — same data, different explainability

### Prerequisites
- Notebook 18_2_1 (pipeline, odds ratios)
- Notebook 18_2_2 (statsmodels, forest plot, individual decomposition)
- Module 18_1_1 and 18_1_2 (confusion matrix, precision/recall, the credit dataset)

## Step 1: Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, ConfusionMatrixDisplay
import statsmodels.api as sm
import xgboost as xgb

sns.set_theme(style='whitegrid')
print('Imports complete.')

## Step 2: Load and Prepare the Data

The South German Credit dataset has 1000 rows and 20 features. The target is whether the applicant is a *good* (0) or *bad* (1) credit risk.

The dataset contains a mix of numerical and categorical features. XGBoost handled these natively in Module 18_1 using `enable_categorical=True`. For logistic regression we need to encode them explicitly.

**Your task:** In the cell below, complete the preprocessing:
1. Load the dataset from OpenML (name='credit-g', version=1)
2. Encode the target: `bad` = 1, `good` = 0
3. One-hot encode categorical features using `pd.get_dummies(drop_first=True)`
4. Cast any boolean columns to int (newer pandas returns bool from `get_dummies`)
5. Create a stratified 70/30 train-test split with `random_state=42`
6. Print the shape of X_train and the class distribution of y_test

In [ ]:
# --- Your Code Here ---
# 1. Load
# data = fetch_openml(name='credit-g', version=1, as_frame=True)
# df = data.frame

# 2. Encode target
# y = ...

# 3. One-hot encode
# X = ...

# 4. Cast bool columns to int

# 5. Train-test split
# X_train, X_test, y_train, y_test = ...

# 6. Print shapes


## Step 3: Build and Train the Pipeline

**Your task:**
1. Identify which columns are continuous (need scaling) and which are binary (don't)
2. Build a `ColumnTransformer` that scales continuous features and passes binary features through using `remainder='passthrough'`
3. Build a `Pipeline` with the preprocessor and a `LogisticRegression` (`max_iter=1000`, `random_state=42`)
4. Fit on the training set and print training and test accuracy

*Hint: the continuous features are the original numeric columns. After one-hot encoding, all new dummy columns are binary and should be passed through.*

In [ ]:
# --- Your Code Here ---


## Step 4: Evaluate

**Your task:**
1. Print the classification report (with target_names=['Good (0)', 'Bad (1)'])
2. Display the confusion matrix
3. Compute and print AUC
4. In a markdown cell below the code, answer: does the model beat the naive baseline? What is the naive baseline accuracy for this dataset?

In [ ]:
# --- Your Code Here ---


*Write your baseline comparison here.*

## Step 5: Odds Ratios with Confidence Intervals

Now fit the same features using `statsmodels` to obtain standard errors and confidence intervals. This is what the compliance team actually needs.

**Your task:**
1. Scale the continuous features manually using `StandardScaler` (fit on train, transform both)
2. Add a constant column with `sm.add_constant()`
3. Fit `sm.Logit(y_train, X_train_sm).fit(disp=0)`
4. Build a DataFrame with columns: feature, OR, OR_low (exp of 2.5th percentile CI), OR_high (exp of 97.5th percentile CI), p_value, significant (p < 0.05)
5. Produce a forest plot (log-scale x-axis, reference line at OR=1, color by significance)

*Hint: Review Notebook 18_2_2, Section 1 — the code structure is identical.*

In [ ]:
# --- Your Code Here ---


### Interpretation

Answer the following in the cell below:

1. Which feature has the largest OR? What does this mean in plain language?
2. Which features have CIs that cross OR=1? What does this mean about those features?
3. The compliance team asks: *'If we deny an applicant, what reasons can we give?'* Based on your forest plot, name the two or three features that provide the strongest evidence for or against creditworthiness.

*Write your interpretation here.*

## Step 6: Explaining an Individual Prediction

Pick a test-set applicant who was predicted as 'bad credit' by your model. Decompose their prediction into feature contributions — the kind of explanation you would provide in an adverse action notice.

**Your task:**
1. Find the index of any applicant in X_test who was predicted as class 1 (bad credit)
2. Compute their predicted probability using the sklearn pipeline
3. Compute the feature contributions (coefficient × scaled feature value) for each feature
4. Print a sorted list of the top contributing features with their direction and magnitude
5. Write 2–3 sentences that could serve as the adverse action explanation for this applicant

*Hint: Review Notebook 18_2_2, Section 3 for the decomposition pattern.*

In [ ]:
# --- Your Code Here ---


*Write the adverse action explanation here.*

## Step 7: Compare LR and XGBoost

Train an XGBoost classifier on the same train/test split. Compare the two models on accuracy, AUC, and — crucially — what each model lets you say about its predictions.

**Your task:**
1. Train `xgb.XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, enable_categorical=True, tree_method='hist', eval_metric='logloss', random_state=42)` on X_train and y_train (use the *original* X_train before dummy encoding if you want to use native categoricals, or use the encoded version — either is fine)
2. Print accuracy and AUC for both models side by side
3. In the markdown cell below, answer: if XGBoost has higher AUC than LR, would you still recommend LR for this credit-scoring use case? Justify your answer.

In [ ]:
# --- Your Code Here ---


*Write your model recommendation and justification here.*

---

## Summary

In this exercise you:

- Built a complete LR pipeline on the credit dataset with proper encoding and scaling
- Produced a forest plot of odds ratios with confidence intervals using statsmodels
- Decomposed an individual prediction into feature contributions for an adverse action notice
- Compared LR and XGBoost under a real-world constraint (interpretability requirement)

The core lesson is that **accuracy is not the only criterion**. When decisions affect people's lives and are subject to legal scrutiny, a slightly less accurate but fully interpretable model is often the right choice.